# NotebookLM Kit — Flashcard Pipeline

Generates one flashcard set per source in a notebook, then downloads them all.

1. **Setup** — load credentials from `.env`
2. **Config** — set notebook ID and output folder
3. **Template** — customise card count, difficulty, and optional prompt
4. **List Sources** — preview what's in the notebook
5. **Create** — submit one flashcard job per source
6. **Poll** — wait for jobs to finish (or resume from disk)
7. **Download** — save every set as JSON

In [45]:
import importlib, pipeline, pipeline._core, pipeline._sources, pipeline._artifacts
importlib.reload(pipeline._core)
importlib.reload(pipeline._sources)
importlib.reload(pipeline._artifacts)
importlib.reload(pipeline)

from pipeline import load_credentials, check_tsx, login, SDK_ROOT

# First time only — opens a browser window for Google login, saves credentials.json:
#   login()

creds = load_credentials(mode="patchright")
check_tsx()

Credentials ready — token: 42 chars, cookies: 2318 chars
tsx: tsx v4.22.3
node v20.17.0


## Config

Find your notebook ID in the NotebookLM URL:  
`https://notebooklm.google.com/notebook/` **`<YOUR_NOTEBOOK_ID>`**

In [46]:
NOTEBOOK_ID = "c7502ffb-bf1c-4a03-8a18-c130d0bfb7e8"  # ← paste your notebook ID here

## Flashcard Template

Optionally provide a focus prompt and adjust card count / difficulty.

In [47]:
# Optional: leave empty to let NotebookLM decide focus automatically
FLASHCARD_INSTRUCTIONS = "Focus on key definitions and any formulas. Avoid trivia."

FLASHCARD_CUSTOMIZATION = {
    "numberOfCards": 1,    # 1 = Fewer cards,  2 = Standard / More cards
    "difficulty":    3,    # 1 = Easy,          2 = Medium,              3 = Hard
    # "language": "en",    # BCP-47 language code — omit to use notebook default
}

## List Sources

In [48]:
from pipeline import list_sources

sources = list_sources(NOTEBOOK_ID, creds)


Found 2 sources:
  [0] 1.md
       id: fcbae21d-987…  status: 2
  [1] 2.md
       id: 36cfe733-fa4…  status: 2


## Create Flashcard Sets

Job IDs are saved to `jobs/flashcards/<timestamp>_jobs.json` — safe to re-run the poll/download cells without re-submitting.

In [50]:
from pipeline import create_artifacts

fc_jobs = create_artifacts(
    NOTEBOOK_ID, "FLASHCARDS", sources,
    FLASHCARD_CUSTOMIZATION, FLASHCARD_INSTRUCTIONS,
    creds,
)

Creating Flashcards for: 1.md
  âœ“ 8c42aab8-0b00-4978-ac22-3435213528bc  state: 1
Creating Flashcards for: 2.md
  âœ“ 32aaa738-38fb-428c-9abd-945dedfcbc0a  state: 1
__JOBS__[{"sourceId":"fcbae21d-987d-4025-b53a-e835f05fec0d","sourceTitle":"1.md","artifactId":"8c42aab8-0b00-4978-ac22-3435213528bc"},{"sourceId":"36cfe733-fa43-47b3-aa1c-40c7b26acd45","sourceTitle":"2.md","artifactId":"32aaa738-38fb-428c-9abd-945dedfcbc0a"}]__JOBS__


✓ Submitted 2 job(s)  →  d:\Core\_Code D\notebooklm-kit\jobs\flashcards\20260530_063607_jobs.json
  1.md  →  8c42aab8-0b00-4978-ac22-3435213528bc
  2.md  →  32aaa738-38fb-428c-9abd-945dedfcbc0a


## Poll Until Ready

Run **the cell below** to wait automatically.  
Or run the **resume cell** first if you're continuing a previous session (jobs already submitted).

In [ ]:
# ── Resume cell — run this instead of Create if you already submitted jobs ───
from pipeline import load_jobs
fc_jobs = load_jobs("FLASHCARDS")

In [52]:
from pipeline import poll_jobs

poll_jobs(fc_jobs, NOTEBOOK_ID, creds, interval=30, max_wait_min=15)

  ✓ 1.md                                                    READY
  ✓ 2.md                                                    READY

✅ All artifacts ready — proceed to download.


True

## Download

Each set is saved as a JSON file in `outputs/flashcards/`.

In [53]:
from pipeline import download_artifacts

download_artifacts(fc_jobs, NOTEBOOK_ID, "FLASHCARDS", creds)

Downloading: 1.md
  âœ“ d:\Core\_Code D\notebooklm-kit\outputs\flashcards\flashcard_foundry_flashcards_1780103226648.json
Downloading: 2.md
  âœ“ d:\Core\_Code D\notebooklm-kit\outputs\flashcards\flashcard_platform_flashcards_1780103230768.json
__RESULTS__[{"sourceTitle":"1.md","filePath":"d:\\Core\\_Code D\\notebooklm-kit\\outputs\\flashcards\\flashcard_foundry_flashcards_1780103226648.json"},{"sourceTitle":"2.md","filePath":"d:\\Core\\_Code D\\notebooklm-kit\\outputs\\flashcards\\flashcard_platform_flashcards_1780103230768.json"}]__RESULTS__


✅ Downloaded 2 / 2 artifact(s)  →  d:\Core\_Code D\notebooklm-kit\outputs\flashcards
  d:\Core\_Code D\notebooklm-kit\outputs\flashcards\20260530_063706_flashcard_foundry_flashcards.json
  d:\Core\_Code D\notebooklm-kit\outputs\flashcards\20260530_063710_flashcard_platform_flashcards.json


[{'sourceTitle': '1.md',
  'filePath': 'd:\\Core\\_Code D\\notebooklm-kit\\outputs\\flashcards\\20260530_063706_flashcard_foundry_flashcards.json'},
 {'sourceTitle': '2.md',
  'filePath': 'd:\\Core\\_Code D\\notebooklm-kit\\outputs\\flashcards\\20260530_063710_flashcard_platform_flashcards.json'}]